# Lesson 16 — Convex Relaxations

## Learning objectives

1. Build the McCormick envelope of a bilinear term {cite:p}`McCormick1976`.
2. Recognize secant relaxations of univariate concave functions.
3. State the αBB convex underestimator {cite:p}`Adjiman1998,Adjiman1998a`.
4. Use `discopt`'s convex-relaxation infrastructure on a small NLP.

## 1. The McCormick envelope

For a bilinear term $z = x y$ with $x \in [\underline x, \overline x], y \in [\underline y, \overline y]$, the **convex hull** is the four McCormick inequalities:

$$
\begin{aligned}
z &\ge \underline x \, y + x \, \underline y - \underline x \, \underline y \\
z &\ge \overline x \, y + x \, \overline y - \overline x \, \overline y \\
z &\le \underline x \, y + x \, \overline y - \underline x \, \overline y \\
z &\le \overline x \, y + x \, \underline y - \overline x \, \underline y
\end{aligned}
$$

These give the *tightest* convex relaxation of the graph $\{(x, y, z) : z = xy\}$ over the box. Bound width determines tightness — narrower boxes → tighter envelopes.

## 2. Secant relaxation for concave $f$

For univariate concave $f$ on $[a, b]$:

- Tightest **convex underestimator**: the secant $f(a) + (f(b) - f(a))(x - a)/(b - a)$.
- Tightest **concave overestimator**: $f$ itself.

For convex $f$, swap the roles. Combined with McCormick for products, this handles **factorable** nonconvex problems {cite:p}`McCormick1976,Smith1999`. A function is *factorable* if it can be expressed as a finite composition of unary operators (exp, log, sin, $\cdot^k$) and binary operators ($+$, $\times$, $/$) applied to the variables; intuitively, anything you would write down by hand. Most engineering objectives and constraints are factorable; the convex-relaxation construction works **term-by-term** on the operator DAG of such a function. We revisit factorability formally in Lesson 22.

## 3. αBB underestimator for general C² functions

For any $f \in C^2$ on a box, the αBB underestimator is

$$\check f(x) = f(x) - \sum_i \alpha_i (\overline x_i - x_i)(x_i - \underline x_i)$$

with $\alpha_i$ large enough to cancel any negative curvature of $f$. Explicit choices use eigenvalue bounds on $\nabla^2 f$ {cite:p}`Adjiman1998`. Convex by construction; tightness depends on $\alpha$.

This is the workhorse for global optimization of general factorable NLPs {cite:p}`Adjiman1998a,Tawarmalani2005`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# Bilinear: min x*y  s.t.  x + y >= 1, x,y in [0, 2].
# discopt automatically applies McCormick relaxations to bilinear terms
# inside its spatial-B&B; you don't construct them manually.
import numpy as np, discopt as do
from discopt.modeling import Model

m = Model("bilinear")
x = m.continuous("x", lb=0, ub=2)
y = m.continuous("y", lb=0, ub=2)
z = m.continuous("z", lb=-10, ub=10)              # auxiliary for the bilinear
m.subject_to(z == x * y)
m.subject_to(x + y >= 1)
m.minimize(z)

r = m.solve()
print(f"global z = {r.objective:.4f}; status: {r.status}; convex fast path: {r.convex_fast_path}")
# (m.relaxation_inequalities() — i.e., the McCormick LP at the root —
#  is not a stable public method; the relaxation lives in the Rust backend.)


## 4. Tightness vs branching

The McCormick relaxation is exact at the box corners; loosest at the center. To tighten: **branch on $x$ (or $y$)** to split the box, recompute envelopes on each child. Spatial B&B (Lesson 21) is exactly this iterated.

The choice of which variable to branch on is the spatial branching rule; common heuristics are *most violating* and *largest box*.

## 5. Trilinear monomials

For products of *three* variables $z = x y w$ on a box, you can build a
McCormick envelope by nesting: pick a pair ($x y$), introduce an
auxiliary $u = x y$ with McCormick on $u$, then McCormick on $u w$. The
catch is that the choice of which pair to nest first changes the envelope
— there are three orderings and they don't agree.

The **permutation-symmetric** trilinear envelope takes the tightest of all
three nested orderings; it is the strongest McCormick-like trilinear
relaxation that depends only on the box {cite:p}`rikun-1997,meyer-floudas-2004`.
A full convex-hull characterization for general box domains has only
recently been completed {cite:p}`makhoul-speakman-2025`.

`discopt` ships both: see `discopt._jax.envelopes.relax_trilinear` (fast,
fixed-order, fallback path) and `relax_trilinear_exact`
(permutation-symmetric). Both run under `jax.vmap` so per-term
relaxations stay batched on the GPU.

## 6. αBB and eigenvalue bounds in `discopt`

The αBB construction needs $\alpha_i$ at least as large as the most
negative eigenvalue of $\nabla^2 f$ over the box. `discopt` computes
*sound* interval-eigenvalue bounds in
`discopt._jax.convexity.eigenvalue` and uses them both for the αBB
underestimator and for emitting a *convexity certificate* — when the
lower eigenvalue bound is $\ge 0$ on the box, the solver can take the
convex fast path (`r.convex_fast_path == True`) and skip spatial
branching entirely.

This is the "M6" piece of the convexification roadmap. The certificate
machinery lives in `discopt._jax.convexity.certificate`; the αBB
arithmetic in `eigenvalue_arith.py`.

## 7. RLT (Reformulation-Linearization Technique)

A more aggressive relaxation: multiply pairs of constraints together
(linear × linear → bilinear, then McCormick on each). Tightens the LP
relaxation at the cost of a quadratic blow-up in size {cite:p}`Sherali1990`.

## References
{cite:p}`McCormick1976,Adjiman1998,Adjiman1998a,Tawarmalani2005,Sherali1990,rikun-1997,meyer-floudas-2004,makhoul-speakman-2025`.